# Daily residual and input diagnostics

Compare the native hybrid ensembles with and without dew point. The inputs shown alongside the residuals are aligned to the mortality dates.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/05_daily_residual_diagnostics.ipynb).

In [ ]:
from pathlib import Path
import sys

repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs" / "training.example.yaml").is_file())
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import pandas as pd
from notebooks.utils import daily_index, ensemble, national_daily
from sari_t.reference import relative_humidity_percent

from notebooks.utils import ANALYSIS_PERIOD, DEVICE, checkpoint_paths
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor


## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to a value from 0 through 9 to inspect another member. The analysis cells evaluate all ten members in a separate inference workspace.

In [ ]:
inspection_member = 0
setups = ['default', 'tas_only']
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Reconstruct daily mortality and align the inputs

In [ ]:
def summarize(model, result):
    daily = national_daily(model, result)
    count = daily["Predicted"].numel()
    inputs = model.data.model_inputs
    tas = inputs.temperature[:, 0, -count:]
    tdew = inputs.temperature[:, 1, -count:]
    daily["Temperature (°C)"] = tas.mean(dim=0)
    daily["Relative humidity (%)"] = relative_humidity_percent(tas, tdew).mean(dim=0)
    population = model.data.population.sum(dim=(1, 3))
    for label, data in [("Non-COVID SARI", inputs.sari), ("COVID-19 SARI", inputs.covid)]:
        daily[label] = (population * data[:, -count:].T).sum(dim=1) / population.sum(dim=1)
    return daily

summaries = {"default": summarize, "tas_only": national_daily}

estimates_by_setup = {}
for setup, summarize_member in summaries.items():
    paths = checkpoints[setup]
    inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
    inference_model = InferenceModel.from_checkpoint(
        paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
    )
    estimates_by_setup[setup] = ensemble(inference_model, paths, summarize_member)
    del inference_model

hybrid = estimates_by_setup["default"]
tas_only = estimates_by_setup["tas_only"]
if not hybrid["Observed"].equal(tas_only["Observed"]):
    raise ValueError("The ensembles must use the same observed mortality labels.")
diagnostics = pd.DataFrame({
    **{name: values.numpy() for name, values in hybrid.items()},
    "With dew point": (hybrid["Observed"] - hybrid["Predicted"]).numpy(),
    "Without dew point": (tas_only["Observed"] - tas_only["Predicted"]).numpy(),
}, index=daily_index())

## Plot residuals, surveillance, and weather

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
diagnostics[["With dew point", "Without dew point"]].plot(ax=axes[0], ylabel="Daily residual")
diagnostics[["Observed", "Predicted"]].plot(ax=axes[1], ylabel="Daily deaths")
diagnostics[["Non-COVID SARI", "COVID-19 SARI"]].plot(ax=axes[2], ylabel="Incidence per 100,000")
diagnostics[["Temperature (°C)", "Relative humidity (%)"]].plot(
    ax=axes[3], secondary_y="Relative humidity (%)", ylabel="Temperature (°C)"
)
axes[3].right_ax.set_ylabel("Relative humidity (%)")
fig.tight_layout()